In [1]:
# Import pandas for handling tabular data
import pandas as pd

# Import NumPy for numerical operations
import numpy as np


# -------- Loading the input file --------

# Read the input gene metadata CSV file
df = pd.read_csv("all_genes_metadata_without_chx.csv")

# Convert CDS length (nucleotides) to protein length (amino acids)
# Divide by 3, round up, and subtract the stop codon
df["Length"] = np.ceil(df["cds_length"] / 3).astype(int) - 1

# Read the MitoCarta annotation file
df_mito = pd.read_excel("mitocarta.xls")


# -------- Standardize --------

# Convert gene names to uppercase and replace missing values with empty strings
df["gene"] = df["gene"].fillna("").str.upper()

# Ensure Length column is numeric
df["Length"] = pd.to_numeric(df["Length"], errors="coerce")

# Standardize MitoCarta gene symbols
df_mito["symbol"] = df_mito["Symbol"].fillna("").str.upper()

# Standardize synonym names
df_mito["synonyms"] = df_mito["Synonyms"].fillna("").str.upper()

# Ensure protein lengths are numeric
df_mito["ProteinLength"] = pd.to_numeric(df_mito["ProteinLength"], errors="coerce")

# Standardize UniProt IDs
df_mito["UniProt"] = df_mito["UniProt"].astype(str).str.upper()


# -------- Build name → proteinLength + UniProt mapping --------

# Dictionary mapping gene names/synonyms to their protein length and UniProt ID
name_to_info = {}

# Iterate through every MitoCarta entry
for _, row in df_mito.iterrows():

    # Get protein length for the current gene
    length = row["ProteinLength"]

    # Get UniProt ID for the current gene
    uid = row["UniProt"]

    # Add the official gene symbol to the dictionary
    if row["symbol"]:
        name_to_info.setdefault(row["symbol"], []).append((length, uid))

    # Add every synonym to the same dictionary
    for s in str(row["synonyms"]).split("|"):
        if s:
            name_to_info.setdefault(s, []).append((length, uid))


# -------- Build length set --------

# Create a set of all mitochondrial protein lengths for fast lookup
mito_lengths = set(df_mito["ProteinLength"].dropna())


# -------- Classification (LOGIC UNCHANGED) --------

# Function to classify each gene based on name and protein length
def classify(row):

    # Extract gene name
    gene = row["gene"]

    # Extract calculated protein length
    length = row["Length"]

    # Check whether gene name exists in MitoCarta
    name_match = gene in name_to_info

    # Check whether protein length exists in MitoCarta
    length_match = length in mito_lengths

    # Variables to store matched protein length and UniProt ID
    mito_len = None
    mito_uid = None

    # If gene name matches, look for an exact protein length match
    if name_match:
        for l, uid in name_to_info[gene]:

            # Exact match of both gene name and protein length
            if pd.notna(length) and length == l:
                mito_len = l
                mito_uid = uid

                # Classified as mitochondrial
                return pd.Series(["mitochondrial", mito_len, mito_uid])

    # Protein length matches but gene name does not
    if length_match:
        return pd.Series(["length_only", None, None])

    # Gene name matches but protein length does not
    elif name_match:
        return pd.Series(["name_only", None, None])

    # Neither name nor length matches
    else:
        return pd.Series(["unclassified", None, None])


# -------- Apply --------

# Apply the classification function to every row
# Create three new columns:
# localization, matched protein length, and UniProt ID
df[["localization", "proteinLength", "UniProt"]] = df.apply(classify, axis=1)


# -------- Save --------
# (Add df.to_csv(...) here if you want to save the annotated table)


# -------- Summary --------

# Print the number of genes in each classification category
print(df["localization"].value_counts())

length_only      8810
unclassified     6433
mitochondrial     836
name_only          91
Name: localization, dtype: int64


In [3]:
# Read the UniProt proteome annotation file
# This file contains the UniProt accession in the "Entry" column
uniprot = pd.read_csv("uniprot_proteome.tsv", sep='\t')


# -------- Standardize --------

# Convert UniProt accession IDs to uppercase and remove extra whitespace
uniprot["Entry"] = uniprot["Entry"].astype(str).str.strip().str.upper()

# Standardize the UniProt IDs from the MitoCarta dataset
df_mito["UniProt"] = df_mito["UniProt"].astype(str).str.strip().str.upper()


# -------- Create set --------

# Create a set of mitochondrial UniProt IDs for fast membership lookup
mito_uniprot_ids = set(df_mito["UniProt"])


# -------- Assign localization --------

# Label each UniProt entry as mitochondrial if it exists in the MitoCarta ID set
# Otherwise label it as non-mitochondrial
uniprot["Localization"] = uniprot["Entry"].apply(
    lambda x: "mitochondrial" if x in mito_uniprot_ids else "non-mitochondrial"
)


# -------- Find missing Mitocarta IDs --------

# Create a set containing all UniProt IDs present in the proteome file
df_ids = set(uniprot["Entry"])

# Find mitochondrial UniProt IDs that are absent from the proteome dataset
missing_ids = mito_uniprot_ids - df_ids

# Print the number of missing mitochondrial UniProt IDs
print("Missing Mitocarta UniProt IDs:", len(missing_ids))

# Print the list of missing UniProt IDs
print(sorted(missing_ids))


# -------- Summary --------

# Print the number of mitochondrial and non-mitochondrial proteins
print(uniprot["Localization"].value_counts())

Missing Mitocarta UniProt IDs: 5
['0', 'B4DEF8', 'B4DJ38', 'P30042', 'Q86UA3']
non-mitochondrial    19288
mitochondrial         1128
Name: Localization, dtype: int64


In [4]:
# Standardize the UniProt IDs in the MitoCarta dataframe
df_mito["UniProt"] = df_mito["UniProt"].astype(str).str.strip().str.upper()

# Convert the missing UniProt ID list into a standardized uppercase set
missing_ids = set(str(x).strip().upper() for x in missing_ids)

# Retrieve information for the missing UniProt IDs from MitoCarta
# Keep only the relevant annotation columns
missing_info = df_mito[
    df_mito["UniProt"].isin(missing_ids)
][["UniProt", "Symbol", "Synonyms", "ProteinLength"]]

# Display the details of the missing mitochondrial proteins
print(missing_info)

     UniProt         Symbol                                 Synonyms  \
119   P30042         GATD3A  C21orf33|ES1|GATD3|GT335|HES1|KNPH|KNPI   
474        0           FMC1                          C7orf55|HSPC268   
605   B4DEF8         MRPL45                            L45mt|MRP-L45   
705        0         CKMT1A                      CKMT1|U-MtCK|mia-CK   
1045  Q86UA3           MYG1        C12orf10|Gamm1|MST024|MSTP024|MYG   
1069       0         CKMT1B                         CKMT|CKMT1|UMTCK   
1071  B4DJ38   ATP5MF-PTCD1                             ATP5J2-PTCD1   
1135       0  RP11_469A15.2                                        -   

      ProteinLength  
119             268  
474             113  
605             306  
705             417  
1045            376  
1069            417  
1071            749  
1135             62  


In [5]:
# Import pandas for DataFrame operations
import pandas as pd


# -------- Step 1: build synonym set ONLY --------

# Create a set to store all synonyms from the missing MitoCarta entries
synonym_set = set()

# Create a dictionary mapping each synonym to its official gene symbol
syn_to_symbol = {}

# Iterate through every missing mitochondrial protein
for _, row in missing_info.iterrows():

    # Get the official gene symbol
    symbol = str(row["Symbol"]).upper()

    # Split the synonym list (pipe-separated) into individual synonyms
    for s in str(row["Synonyms"]).upper().split("|"):

        # Remove leading/trailing whitespace
        s = s.strip()

        # Ignore empty values and placeholder "-"
        if s and s != "-":

            # Add synonym to the lookup set
            synonym_set.add(s)

            # Store mapping: synonym → official gene symbol
            syn_to_symbol[s] = symbol


# -------- Step 2: normalize df --------

# Standardize gene names in the input dataframe
df["gene"] = df["gene"].astype(str).str.upper()


# -------- Step 3: update localization + gene --------

# Function to update localization using synonym matches
def update_loc(row):

    # Only recheck genes previously classified as partial matches
    if row["localization"] in ["name_only", "length_only"]:

        # Get the gene name
        gene = row["gene"]

        # If the gene matches one of the mitochondrial synonyms
        if gene in synonym_set:

            # Update localization to mitochondrial
            row["localization"] = "mitochondrial"

            # Replace the synonym with the official gene symbol
            row["gene"] = syn_to_symbol.get(gene, gene)

    # Return the updated row
    return row


# Apply the update function to every row of the dataframe
df = df.apply(update_loc, axis=1)


# -------- Summary --------

# Print the updated counts for each localization category
print(df["localization"].value_counts())

length_only      8809
unclassified     6433
mitochondrial     837
name_only          91
Name: localization, dtype: int64


In [6]:
# Import pandas for DataFrame operations
import pandas as pd


# -------- Step 1: split UniProt --------

# Create a dataframe containing only mitochondrial proteins
uni_mito = uniprot[
    uniprot["Localization"] == "mitochondrial"
].copy()

# Create a dataframe containing all non-mitochondrial proteins
uni_non_mito = uniprot[
    uniprot["Localization"] != "mitochondrial"
].copy()


# -------- Step 2: normalize --------

# Function to standardize gene names
# Converts text to uppercase and removes surrounding whitespace
def normalize_gene(x):
    return str(x).upper().strip()


# Standardize both mitochondrial and non-mitochondrial dataframes
for d in [uni_mito, uni_non_mito]:

    # Normalize primary gene names
    d["Gene Names"] = d["Gene Names"].apply(normalize_gene)

    # Normalize synonym names and replace missing values
    d["Gene Names (synonym)"] = (
        d["Gene Names (synonym)"]
        .fillna("")
        .apply(normalize_gene)
    )

    # Ensure protein length is numeric
    d["Length"] = pd.to_numeric(
        d["Length"],
        errors="coerce"
    )

    # Standardize UniProt accession IDs
    d["Entry"] = (
        d["Entry"]
        .astype(str)
        .str.strip()
        .str.upper()
    )


# -------- Step 3: subset --------

# Select genes that are still not confidently classified
df_subset = df[
    df["localization"].isin(
        ["name_only", "length_only", "unclassified"]
    )
].copy()

# Standardize gene names
df_subset["gene"] = df_subset["gene"].apply(normalize_gene)

# Ensure protein lengths are numeric
df_subset["Length"] = pd.to_numeric(
    df_subset["Length"],
    errors="coerce"
)


# -------- Step 4: build lookup --------

# Build a lookup dictionary using gene names and synonyms
def build_lookup(df_in):

    # Dictionary mapping gene names/synonyms to canonical gene,
    # protein length, and UniProt accession
    lookup = {}

    # Iterate through every UniProt record
    for _, row in df_in.iterrows():

        # Get protein length
        length = row["Length"]

        # Get UniProt accession
        entry = row["Entry"]

        # Split primary gene names into a list
        genes = row["Gene Names"].split()

        # Skip records with no gene names
        if len(genes) == 0:
            continue

        # Use the first gene name as the canonical symbol
        canonical_gene = genes[0]

        # Add all primary gene names to the lookup dictionary
        for g in genes:

            lookup.setdefault(g, []).append(
                (canonical_gene, length, entry)
            )

        # Add all synonyms to the lookup dictionary
        for s in row["Gene Names (synonym)"].split():

            if s:

                lookup.setdefault(s, []).append(
                    (canonical_gene, length, entry)
                )

    # Return the completed lookup dictionary
    return lookup


# Build lookup dictionary for mitochondrial proteins
mito_lookup = build_lookup(uni_mito)

# Build lookup dictionary for non-mitochondrial proteins
non_mito_lookup = build_lookup(uni_non_mito)


# -------- Step 5: classify --------

# Classify genes using UniProt gene names and protein lengths
def classify(row):

    # Extract gene name and protein length
    gene = row["gene"]
    length = row["Length"]

    # Retrieve possible mitochondrial matches
    mito_matches = mito_lookup.get(gene, [])

    # Retrieve possible non-mitochondrial matches
    non_mito_matches = non_mito_lookup.get(gene, [])

    # ---------- mitochondrial ----------

    # Check for an exact mitochondrial name + length match
    for uni_gene, uni_len, entry in mito_matches:

        if pd.notna(length) and length == uni_len:

            # Update localization
            row["localization"] = "mitochondrial"

            # Store matched protein length
            row["proteinLength"] = uni_len

            # Store matched UniProt accession
            row["Uniprot"] = entry

            # Return updated row immediately
            return row

    # ---------- non-mito ----------

    # Check for an exact non-mitochondrial name + length match
    for uni_gene, uni_len, entry in non_mito_matches:

        if pd.notna(length) and length == uni_len:

            # Update localization
            row["localization"] = "non-mitochondrial"

            # Store matched protein length
            row["proteinLength"] = uni_len

            # Store matched UniProt accession
            row["Uniprot"] = entry

            # Return updated row immediately
            return row

    # ---------- no match ----------

    # Return the original row if no exact match is found
    return row


# -------- Step 6: apply --------

# Apply the classification function to all unresolved genes
df_subset = df_subset.apply(
    classify,
    axis=1
)


# -------- Step 7: merge back --------

# Update the original dataframe with all modified columns
# except the temporary UniProt column
df.update(
    df_subset.drop(
        columns=["Uniprot"],
        errors="ignore"
    )
)

# Copy the temporary UniProt accessions into the original dataframe
df.loc[df_subset.index, "Uniprot"] = df_subset["Uniprot"]

# Merge newly assigned UniProt IDs with existing ones
# Existing IDs are kept when no new assignment exists
df["UniProt"] = df["Uniprot"].combine_first(df["UniProt"])

# Remove the temporary UniProt column
df.drop(columns=["Uniprot"], inplace=True)


# -------- Summary --------

# Display the final number of genes in each localization category
print(df["localization"].value_counts())

non-mitochondrial    9795
length_only          2787
unclassified         2649
mitochondrial         867
name_only              72
Name: localization, dtype: int64


In [7]:
# Function to fill missing UniProt IDs using gene name and protein length
def fill_uniprot_by_name_length(df, uniprot_df):

    # Helper function to standardize text
    # Converts text to uppercase and removes surrounding whitespace
    def norm(x):
        return str(x).upper().strip()

    # Create a copy so the original UniProt dataframe is not modified
    u = uniprot_df.copy()

    # Standardize UniProt accession IDs
    u["Entry"] = u["Entry"].astype(str).str.strip().str.upper()

    # Ensure protein lengths are numeric
    u["Length"] = pd.to_numeric(u["Length"], errors="coerce")

    # Standardize primary gene names
    u["Gene Names"] = u["Gene Names"].fillna("").apply(norm)

    # Standardize gene synonyms
    u["Gene Names (synonym)"] = u["Gene Names (synonym)"].fillna("").apply(norm)

    # Build lookup dictionary:
    # (gene name, protein length) → [(UniProt ID, canonical gene name)]
    lookup = {}

    # Iterate through every UniProt record
    for _, r in u.iterrows():

        # Get protein length
        L = r["Length"]

        # Skip records without a valid length
        if pd.isna(L):
            continue

        # Get UniProt accession
        entry = r["Entry"]

        # Split primary gene names
        genes = r["Gene Names"].split()

        # Use the first gene name as the canonical symbol
        canonical = genes[0] if genes else None

        # Add all primary gene names to the lookup dictionary
        for g in genes:
            lookup.setdefault((g, L), []).append((entry, canonical))

        # Add all synonyms to the lookup dictionary
        for s in r["Gene Names (synonym)"].split():
            if s:
                lookup.setdefault((s, L), []).append((entry, canonical))

    # Create a copy of the input dataframe
    d = df.copy()

    # Standardize gene names
    d["gene"] = d["gene"].apply(norm)

    # Ensure protein lengths are numeric
    d["Length"] = pd.to_numeric(d["Length"], errors="coerce")

    # Standardize UniProt IDs
    d["UniProt"] = d["UniProt"].astype(str).str.strip().str.upper()

    # Identify rows with missing or invalid UniProt IDs
    missing = d["UniProt"].isin(["", "0", "NAN"]) | d["UniProt"].isna()

    # Iterate through rows needing UniProt assignment
    for idx in d.index[missing]:

        # Retrieve gene name and protein length
        g = d.at[idx, "gene"]
        L = d.at[idx, "Length"]

        # Skip rows with missing gene name or length
        if pd.isna(g) or pd.isna(L):
            continue

        # Find matching UniProt entries using gene name and length
        hits = lookup.get((g, L), [])

        # Skip if no match exists
        if not hits:
            continue

        # Use the first matching UniProt record
        # (Additional tie-breaking rules can be added if necessary)
        entry, canonical = hits[0]

        # Fill the missing UniProt accession
        d.at[idx, "UniProt"] = entry

        # Replace the gene name with the canonical UniProt gene symbol
        if canonical:
            d.at[idx, "gene"] = canonical

        # Record the matched protein length
        d.at[idx, "proteinLength"] = L

    # Return the updated dataframe
    return d


# Step A: Fill missing UniProt IDs using gene name and protein length
df = fill_uniprot_by_name_length(df, uniprot)

In [8]:
# Function to assign protein sequences using UniProt accession IDs
def assign_sequence_from_uniprot(df, uniprot_df, seq_col="Sequence"):

    # -------------------------
    # 1. build lookup: Entry → Sequence
    # -------------------------

    # Create a dictionary mapping UniProt accession IDs
    # to their corresponding protein sequences
    seq_map = (
        uniprot_df.dropna(subset=["Entry", seq_col])
        .assign(
            Entry=lambda x: x["Entry"]
            .astype(str)
            .str.strip()
            .str.upper()
        )
        .set_index("Entry")[seq_col]
        .to_dict()
    )

    # -------------------------
    # 2. normalize UniProt column in df
    # -------------------------

    # Standardize UniProt accession IDs in the input dataframe
    df["UniProt"] = (
        df["UniProt"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    # -------------------------
    # 3. assign sequence
    # -------------------------

    # Assign the protein sequence corresponding to each UniProt ID
    # Rows without a matching UniProt accession will receive NaN
    df["sequence"] = df["UniProt"].map(seq_map)

    # Return the updated dataframe
    return df


# Assign protein sequences to the dataframe using UniProt IDs
df = assign_sequence_from_uniprot(df, uniprot)

In [9]:
# Select rows where the protein sequence is missing
# (either NaN or an empty string)
missing_seq = df[df["sequence"].isna() | (df["sequence"] == "")]

# Keep only mitochondrial proteins with missing sequences
mito_missing = missing_seq[
    missing_seq["localization"] == "mitochondrial"
]

# Check whether each mitochondrial UniProt ID exists
# in the UniProt reference dataframe
mito_missing["exists_in_uniprot"] = mito_missing["UniProt"].isin(
    uniprot["Entry"]
)

# Print the total number of mitochondrial proteins
# lacking sequences
print("Mito total missing:", len(mito_missing))

# Print how many of those UniProt IDs are present
# in the UniProt reference file
print(
    "Mito found in UniProt:",
    mito_missing["exists_in_uniprot"].sum()
)

# Keep only non-mitochondrial proteins with missing sequences
non_mito_missing = missing_seq[
    missing_seq["localization"] == "non-mitochondrial"
]

# Check whether each non-mitochondrial UniProt ID exists
# in the UniProt reference dataframe
non_mito_missing["exists_in_uniprot"] = non_mito_missing["UniProt"].isin(
    uniprot["Entry"]
)

# Print the total number of non-mitochondrial proteins
# lacking sequences
print("Non-mito total missing:", len(non_mito_missing))

# Print how many of those UniProt IDs are present
# in the UniProt reference file
print(
    "Non-mito found in UniProt:",
    non_mito_missing["exists_in_uniprot"].sum()
)

Mito total missing: 5
Mito found in UniProt: 0
Non-mito total missing: 0
Non-mito found in UniProt: 0


/var/folders/v1/6kddl0v12td99m1lchv02v4m0000gn/T/ipykernel_9928/3919644955.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mito_missing["exists_in_uniprot"] = mito_missing["UniProt"].isin(


In [10]:
# Identify rows where the protein sequence is missing
# Missing values include NaN, empty strings, "None", and "nan"
missing_idx = df[
    df["sequence"].isna() |
    df["sequence"].astype(str).str.strip().str.lower().isin(
        ["", "none", "nan"]
    )
].index

# Print the number of proteins that still have no sequence
print(f"Seq not found in df: {len(missing_idx)}")


# Import the requests library for making HTTP requests
import requests


# Function to retrieve a protein sequence from UniProt
# using its UniProt accession ID
def fetch_uniprot_seq(uid):

    try:
        # Construct the UniProt REST API URL
        url = f"https://rest.uniprot.org/uniprotkb/{uid}.json"

        # Send a GET request with a 15-second timeout
        r = requests.get(url, timeout=15)

        # Return None if the UniProt accession was not found
        if r.status_code != 200:
            return None, None

        # Parse the JSON response
        data = r.json()

        # Extract the amino acid sequence
        seq = data["sequence"]["value"]

        # Extract the sequence length
        length = data["sequence"]["length"]

        # Return both sequence and length
        return seq, length

    # Return None values if any error occurs
    except:
        return None, None

Seq not found in df: 5513


In [11]:
# Iterate through every row with a missing protein sequence
for idx in missing_idx:

    # Retrieve and standardize the UniProt accession ID
    uid = str(df.at[idx, "UniProt"]).upper()

    # Skip rows with missing or invalid UniProt IDs
    if uid in ["", "NONE", "NAN"]:
        continue

    # Fetch the protein sequence and its length from UniProt
    seq, length = fetch_uniprot_seq(uid)

    # Skip if the UniProt record could not be retrieved
    if seq is None:
        continue

    # Get the expected protein length from the dataframe
    expected_len = df.at[idx, "Length"]

    # Assign the sequence only if the retrieved sequence length
    # matches the expected protein length
    if pd.notna(expected_len) and int(length) == int(expected_len):
        df.at[idx, "sequence"] = seq

In [12]:
# Identify rows where the protein sequence is still missing
# Missing values include NaN, empty strings, "None", and "nan"
seq = df[
    df["sequence"].isna() |
    df["sequence"].astype(str).str.strip().str.lower().isin(
        ["", "none", "nan"]
    )
].index

# Print the number of proteins that still have no assigned sequence
print(f"Seq not found in df: {len(seq)}")

Seq not found in df: 5511


In [13]:
# Identify genes whose names contain a hyphen ("-")
# and classify them as readthrough genes
df.loc[
    df["gene"].str.contains("-", na=False),
    "localization"
] = "readthrough"

In [14]:
# Create a new dataframe containing only genes that remain
# partially classified or completely unclassified
data = df[
    df["localization"].isin(
        ["name_only", "length_only", "unclassified"]
    )
].copy()

In [15]:
# Import Biopython modules for accessing NCBI and parsing FASTA records
from Bio import Entrez, SeqIO

# Import pandas for tabular data manipulation
import pandas as pd

# Import time for adding delays between API requests
import time

# Import requests (not used in this script but available if needed)
import requests


# ── Set your API key ──────────────────────────────────────────────────────────

# Email address required by the NCBI Entrez service
Entrez.email = "your@gmail.com"

# NCBI API key for higher request limits
Entrez.api_key = "NCBI_API_KEY"


# -------------------------------------
# Step 1 — get unique protein IDs
# -------------------------------------

# Extract all unique protein accession IDs from the dataframe
protein_ids = data["protein_id"].dropna().unique().tolist()

# Print the total number of proteins to retrieve
print("Total proteins:", len(protein_ids))


# -------------------------------------
# Step 2 — batch fetch with retry
# -------------------------------------

# Function to fetch protein sequences from NCBI in batches
def fetch_batch(batch, retries=3):

    # Retry the request if it fails
    for attempt in range(retries):

        try:

            # Request FASTA records for the current batch of protein IDs
            handle = Entrez.efetch(
                db="protein",
                id=",".join(batch),
                rettype="fasta",
                retmode="text"
            )

            # Parse the returned FASTA records
            records = list(SeqIO.parse(handle, "fasta"))

            # Close the network handle
            handle.close()

            # Return the retrieved protein records
            return records

        except Exception as e:

            # Print the error message for the failed attempt
            print(f"  Attempt {attempt+1} failed: {e}")

            # Wait longer before retrying
            time.sleep(5 * (attempt + 1))

    # Print a message if all retry attempts fail
    print(f"  Giving up on batch starting at {batch[0]}")

    # Return an empty list if retrieval was unsuccessful
    return []


# Number of protein IDs to retrieve per request
batch_size = 200

# List for storing retrieved protein sequences
seq_rows = []


# Retrieve protein sequences batch by batch
for i in range(0, len(protein_ids), batch_size):

    # Select the current batch of protein IDs
    batch = protein_ids[i:i + batch_size]

    # Fetch the protein sequences
    records = fetch_batch(batch)

    # Store the retrieved sequences
    for rec in records:

        seq_rows.append({

            # Protein accession ID
            "protein_id": rec.id,

            # Amino acid sequence
            "sequence": str(rec.seq),

        })

    # Pause briefly to stay within the API rate limit
    time.sleep(0.11)

    # Print retrieval progress
    print(
        f"  Fetched {min(i + batch_size, len(protein_ids))}/{len(protein_ids)}"
    )


# Convert the retrieved sequence list into a dataframe
df_seq = pd.DataFrame(seq_rows)

# Display the first few retrieved protein sequences
print(df_seq.head())

Total proteins: 5422
  Fetched 200/5422
  Fetched 400/5422
  Fetched 600/5422
  Fetched 800/5422
  Fetched 1000/5422
  Fetched 1200/5422
  Fetched 1400/5422
  Fetched 1600/5422
  Fetched 1800/5422
  Fetched 2000/5422
  Fetched 2200/5422
  Fetched 2400/5422
  Fetched 2600/5422
  Fetched 2800/5422
  Fetched 3000/5422
  Fetched 3200/5422
  Fetched 3400/5422
  Fetched 3600/5422
  Fetched 3800/5422
  Fetched 4000/5422
  Fetched 4200/5422
  Fetched 4400/5422
  Fetched 4600/5422
  Fetched 4800/5422
  Fetched 5000/5422
  Fetched 5200/5422
  Fetched 5400/5422
  Fetched 5422/5422
       protein_id                                           sequence
0  XP_006719119.1  MGKNKLLHPSLVLLLLVLLPTDASVSGKPQYMVLVPSLLHTETTEK...
1  XP_011518868.1  MWAQLLLGMLALSPAIAEELPNYLVTLPARLNFPSVQKVCLDLSPG...
2  XP_011537080.1  MCSLGLFPPPPPRGQVTLYEHNNELVTGSSYESPPPDFRGQWINLP...
3  XP_006714294.1  MLKFVLVAGMLPRRVPRVSLTKDGPQPLKKKLDAWLSEDMNYARFI...
4  NP_001378962.1  MTSPEIASLSWGQMKVKGSNTTYKDCKVWPGGSRTWDWRETGTESL...


In [16]:
# Set the email required by the NCBI Entrez service
Entrez.email = "your@gmail.com"

# Set the NCBI API key for higher request limits
Entrez.api_key = "NCBI_API_KEY"
# Extract all unique protein accession IDs from the dataframe
protein_ids = df_seq["protein_id"].dropna().unique().tolist()

# Print the total number of proteins to process
print("Total proteins:", len(protein_ids))


# Function to fetch Gene IDs corresponding to a batch of protein IDs
def fetch_gene_ids(batch, retries=3):

    # Retry failed requests up to the specified number of times
    for attempt in range(retries):

        try:

            # Download GenBank records for the batch of protein IDs
            handle = Entrez.efetch(
                db="protein",
                id=",".join(batch),
                rettype="gb",
                retmode="text"
            )

            # Parse the downloaded GenBank records
            records = list(SeqIO.parse(handle, "gb"))

            # Close the network connection
            handle.close()

            # Store extracted Gene IDs
            rows = []

            # Process each protein record
            for rec in records:

                # Initialize Gene ID as missing
                gene_id = None

                # Search through all annotated features
                for feature in rec.features:

                    # Examine database cross-references
                    for xref in feature.qualifiers.get("db_xref", []):

                        # Look for the NCBI GeneID entry
                        if xref.startswith("GeneID:"):

                            # Extract the numeric Gene ID
                            gene_id = xref.split(":")[1]

                            # Stop searching once Gene ID is found
                            break

                    # Exit the feature loop if Gene ID has been found
                    if gene_id:
                        break

                # Store the protein ID and its Gene ID
                rows.append({
                    "protein_id": rec.id,
                    "gene_id": gene_id
                })

            # Return the extracted Gene IDs for this batch
            return rows

        except Exception as e:

            # Print the error message if the request fails
            print(f"  Attempt {attempt+1} failed: {e}")

            # Wait progressively longer before retrying
            time.sleep(5 * (attempt + 1))

    # Print a message if all retry attempts fail
    print(f"  Giving up on batch starting at {batch[0]}")

    # Return an empty list if retrieval failed
    return []


# Number of protein IDs to retrieve per request
batch_size = 200

# List for storing extracted Gene IDs
gene_rows = []


# Process all protein IDs in batches
for i in range(0, len(protein_ids), batch_size):

    # Select the current batch
    batch = protein_ids[i:i + batch_size]

    # Retrieve Gene IDs for the current batch
    gene_rows.extend(fetch_gene_ids(batch))

    # Pause briefly to respect the API rate limit
    time.sleep(0.11)

    # Print progress
    print(
        f"  Fetched {min(i + batch_size, len(protein_ids))}/{len(protein_ids)}"
    )


# Convert the Gene ID records into a dataframe
df_gene_id = pd.DataFrame(gene_rows)

# Merge the Gene IDs back into the protein sequence dataframe
df_seq = df_seq.merge(
    df_gene_id,
    on="protein_id",
    how="left"
)

# Display the first ten merged records
print(df_seq.head(10))


Total proteins: 5422
  Fetched 200/5422
  Fetched 400/5422
  Fetched 600/5422
  Fetched 800/5422
  Fetched 1000/5422
  Fetched 1200/5422
  Fetched 1400/5422
  Fetched 1600/5422
  Fetched 1800/5422
  Fetched 2000/5422
  Fetched 2200/5422
  Fetched 2400/5422
  Fetched 2600/5422
  Fetched 2800/5422
  Fetched 3000/5422
  Fetched 3200/5422
  Fetched 3400/5422
  Fetched 3600/5422
  Fetched 3800/5422
  Fetched 4000/5422
  Fetched 4200/5422
  Fetched 4400/5422
  Fetched 4600/5422
  Fetched 4800/5422
  Fetched 5000/5422
  Fetched 5200/5422
  Fetched 5400/5422
  Fetched 5422/5422
       protein_id                                           sequence gene_id
0  XP_006719119.1  MGKNKLLHPSLVLLLLVLLPTDASVSGKPQYMVLVPSLLHTETTEK...       2
1  XP_011518868.1  MWAQLLLGMLALSPAIAEELPNYLVTLPARLNFPSVQKVCLDLSPG...  144568
2  XP_011537080.1  MCSLGLFPPPPPRGQVTLYEHNNELVTGSSYESPPPDFRGQWINLP...    8086
3  XP_006714294.1  MLKFVLVAGMLPRRVPRVSLTKDGPQPLKKKLDAWLSEDMNYARFI...   51166
4  NP_001378962.1  MTSPEIASLSWGQMKVKGS

In [17]:
df_seq[
    df_seq["gene_id"].isna() |
    (df_seq["gene_id"].astype(str).str.strip() == "")
]

,protein_id,sequence,gene_id


In [18]:
# Import requests for interacting with the UniProt REST API
import requests

# Import pandas for DataFrame operations
import pandas as pd

# Import time for polling delays
import time

# Import StringIO for reading TSV text directly into pandas
from io import StringIO


# Base URL for the UniProt ID Mapping REST API
IDMAP_BASE = "https://rest.uniprot.org/idmapping"


# Function to map identifiers (e.g., GeneID → UniProtKB)
# Supports automatic polling and pagination
def run_idmapping(ids, from_db="GeneID", to_db="UniProtKB"):

    """Batch ID mapping with full pagination support."""

    # ------------------------------------------------------------------
    # Step 1: Submit the mapping job
    # ------------------------------------------------------------------

    # Send the list of IDs to the UniProt mapping service
    r = requests.post(
        f"{IDMAP_BASE}/run",
        data={
            "from": from_db,
            "to": to_db,
            "ids": ",".join(map(str, ids))
        },
        timeout=30
    )

    # Raise an exception if the request failed
    r.raise_for_status()

    # Retrieve the job ID assigned by UniProt
    job_id = r.json()["jobId"]

    # Print the submitted job ID
    print(f"Job submitted: {job_id}")


    # ------------------------------------------------------------------
    # Step 2: Poll the job until it finishes
    # ------------------------------------------------------------------

    # Check job status every 3 seconds (maximum 60 attempts)
    for _ in range(60):

        time.sleep(3)

        # Query the current job status
        status = requests.get(
            f"{IDMAP_BASE}/status/{job_id}",
            timeout=15
        )

        # Raise an exception if the request failed
        status.raise_for_status()

        # Parse the returned JSON
        js = status.json()

        # Stop polling once results are available
        if "results" in js or "failedIds" in js:
            print("Job complete.")
            break

        # Stop immediately if the job failed
        if js.get("jobStatus") == "FAILURE":
            raise RuntimeError(f"Job failed: {js}")

    # Raise an error if polling timed out
    else:
        raise TimeoutError("Job timed out.")


    # ------------------------------------------------------------------
    # Step 3: Download all result pages
    # ------------------------------------------------------------------

    # List for storing all retrieved mapping tables
    all_rows = []

    # Initial results URL
    url = f"{IDMAP_BASE}/uniprotkb/results/{job_id}"

    # Request up to 500 mappings per page
    params = {
        "fields": "accession,reviewed",
        "format": "tsv",
        "size": 500
    }

    # Continue downloading until no additional pages remain
    while url:

        # Download one page of results
        res = requests.get(
            url,
            params=params,
            timeout=60
        )

        # Raise an exception if the request failed
        res.raise_for_status()

        # Split the returned TSV into lines
        lines = res.text.strip().splitlines()

        # Ignore empty result pages
        if len(lines) > 1:

            # Read the TSV page into a dataframe
            df_page = pd.read_csv(
                StringIO(res.text),
                sep="\t"
            )

            # Store the page
            all_rows.append(df_page)

        # Check whether another page exists
        link_header = res.headers.get("Link", "")

        if 'rel="next"' in link_header:

            # Extract the URL of the next page
            url = link_header.split("<")[1].split(">")[0]

            # Parameters are already embedded in the next-page URL
            params = None

        else:

            # Stop once all pages have been downloaded
            url = None


    # ------------------------------------------------------------------
    # Step 4: Handle empty results
    # ------------------------------------------------------------------

    # Return an empty dataframe if no mappings were found
    if not all_rows:
        return pd.DataFrame(
            columns=["gene_id", "uniprot_id"]
        )


    # ------------------------------------------------------------------
    # Step 5: Combine and clean the mapping results
    # ------------------------------------------------------------------

    # Merge all downloaded pages
    df = pd.concat(
        all_rows,
        ignore_index=True
    )

    # Rename UniProt output columns
    df = df.rename(
        columns={
            "From": "gene_id",
            "Entry": "uniprot_id",
            "Reviewed": "reviewed"
        }
    )

    # Keep only the required columns and remove duplicates
    df = df[
        ["gene_id", "uniprot_id", "reviewed"]
    ].drop_duplicates()


    # ------------------------------------------------------------------
    # Step 6: Prefer reviewed UniProt entries
    # ------------------------------------------------------------------

    # Assign priority:
    # Reviewed (Swiss-Prot) = 0
    # Unreviewed (TrEMBL) = 1
    df["rank"] = df["reviewed"].apply(
        lambda x: 0 if str(x).lower() == "reviewed" else 1
    )

    # Keep only the highest-priority mapping for each Gene ID
    df = (
        df.sort_values("rank")
          .drop_duplicates(subset="gene_id", keep="first")
    )

    # Remove helper columns
    df = df.drop(columns=["rank", "reviewed"])

    # Return the cleaned mapping table
    return df


# ── Run ───────────────────────────────────────────────────────────────

# Collect all unique Gene IDs from the dataframe
gene_ids = (
    df_seq["gene_id"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

# Print the number of Gene IDs to be mapped
print(f"Mapping {len(gene_ids)} Gene IDs...")

# Perform GeneID → UniProtKB mapping
geneid_to_uniprot = run_idmapping(gene_ids)

# Print the number of successfully mapped Gene IDs
print(f"Resolved: {len(geneid_to_uniprot)}")

# Display the first few mappings
print(geneid_to_uniprot.head())

Mapping 5422 Gene IDs...
Job submitted: 5xVaeTzvKp
Job complete.
Resolved: 5347
      gene_id uniprot_id
0           2     P01023
6676     4678     P49321
6675    26502     Q9UHQ1
6674    93100     Q6XQN6
6672     8774     Q99747


In [21]:
# Identify UniProt IDs that are mapped from multiple Gene IDs
geneid_to_uniprot[
    geneid_to_uniprot["uniprot_id"].duplicated(keep=False)
]

# Sort the duplicated mappings by UniProt ID for easier inspection
geneid_to_uniprot.sort_values("uniprot_id")

,gene_id,uniprot_id
1294,400073,A0A087WX00
9997,222183,A0A087WXA3
9543,146429,A0A087WY01
8365,374462,A0A087WZU1
3157,388795,A0A096LNH2
...,...,...
6591,23026,Q9Y6X6
6351,22880,Q9Y6X9
5093,10625,Q9Y6Y0
1511,23261,Q9Y6Y1


In [22]:
# Remove any existing UniProt ID columns to avoid duplicate columns after merging
df_seq = df_seq.drop(
    columns=[
        c for c in df_seq.columns
        if "uniprot_id" in c
    ]
)

# Convert Gene IDs in df_seq to strings for consistent merging
df_seq["gene_id"] = df_seq["gene_id"].astype(str)

# Convert Gene IDs in the mapping table to strings
geneid_to_uniprot["gene_id"] = geneid_to_uniprot["gene_id"].astype(str)

# Merge the GeneID → UniProt mapping into the sequence dataframe
# Keep all rows from df_seq even if no UniProt mapping exists
df_seq = df_seq.merge(
    geneid_to_uniprot[
        ["gene_id", "uniprot_id"]
    ],
    on="gene_id",
    how="left"
)

# Display the first few rows showing the protein ID,
# Gene ID, and mapped UniProt accession
print(
    df_seq[
        ["protein_id", "gene_id", "uniprot_id"]
    ].head()
)

       protein_id gene_id uniprot_id
0  XP_006719119.1       2     P01023
1  XP_011518868.1  144568     A8K2U0
2  XP_011537080.1    8086     Q9NRG9
3  XP_006714294.1   51166     Q8N5Z0
4  NP_001378962.1   28971     Q9H7C9


In [23]:
# Remove the existing sequence column from the dataframe if it exists
# This avoids duplicate sequence columns during the merge
data = data.drop(columns=["sequence"], errors="ignore")

# Calculate the amino acid sequence length for each retrieved protein
df_seq["seq_length"] = df_seq["sequence"].str.len()

# Merge sequence information into the protein dataframe
# Matching is performed using the protein accession ID
temp = data.merge(
    df_seq[
        [
            "sequence",
            "gene_id",
            "protein_id",
            "seq_length",
            "uniprot_id"
        ]
    ],
    on="protein_id",
    how="left"
)

# Keep proteins whose expected length matches the retrieved sequence length
matched_df = temp[
    temp["Length"] == temp["seq_length"]
]

# Collect proteins where the expected and retrieved lengths differ
unmatched_df = temp[
    temp["Length"] != temp["seq_length"]
]

# Report whether all sequences matched the expected lengths
if unmatched_df.empty:

    # No length mismatches were found
    print("All matched")

else:

    # Print the number of proteins with length mismatches
    print(f"{len(unmatched_df)} unmatched rows")

30 unmatched rows


In [24]:
matched_df["gene_id"] = (
    matched_df["gene_id"]
    .astype(float)
    .astype(int)
    
)


/var/folders/v1/6kddl0v12td99m1lchv02v4m0000gn/T/ipykernel_9928/943418394.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  matched_df["gene_id"] = (


In [25]:
# Build a lookup table containing UniProt accession IDs and their localization
# from the reference UniProt dataframe
uniprot_loc = uniprot[["Entry", "Localization"]].copy()

# Standardize UniProt accession IDs by converting them to uppercase
# and removing leading/trailing whitespace
uniprot_loc["Entry"] = (
    uniprot_loc["Entry"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Remove duplicate UniProt accessions, keeping the first occurrence
uniprot_loc = uniprot_loc.drop_duplicates("Entry")


# Normalize the mapped UniProt IDs in matched_df
matched_df["uniprot_id"] = (
    matched_df["uniprot_id"]
    .astype(str)
    .str.strip()
    .str.upper()
)


# Merge localization information from the UniProt reference table
# using the mapped UniProt accession IDs
matched_df = matched_df.merge(
    uniprot_loc,
    left_on="uniprot_id",
    right_on="Entry",
    how="left"
)


# Identify rows where:
# 1. A UniProt localization is available, and
# 2. The current localization is still unresolved
mask = (
    matched_df["Localization"].notna() &
    matched_df["localization"].isin(
        ["name_only", "length_only", "unclassified"]
    )
)


# Update localization based on the UniProt reference annotation
matched_df.loc[mask, "localization"] = (
    matched_df.loc[mask, "Localization"]
    .apply(
        lambda x:
        "mitochondrial"
        if str(x).lower() == "mitochondrial"
        else "non-mitochondrial"
    )
)


# Fill missing UniProt IDs using the mapped UniProt accession
# only when the current value is "NONE"
matched_df.loc[
    mask & (matched_df["UniProt"] == "NONE"),
    "UniProt"
] = matched_df.loc[
    mask & (matched_df["UniProt"] == "NONE"),
    "uniprot_id"
]


# Remove temporary columns created during the merge
matched_df.drop(
    columns=["Entry", "Localization"],
    inplace=True
)


# Display the updated localization counts
print(matched_df["localization"].value_counts())

non-mitochondrial    5061
mitochondrial         220
length_only            70
unclassified           41
Name: localization, dtype: int64


/var/folders/v1/6kddl0v12td99m1lchv02v4m0000gn/T/ipykernel_9928/836852993.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  matched_df["uniprot_id"] = (


In [26]:
# Create a dataframe containing proteins whose localization
# is still unresolved after UniProt-based annotation
non_uniprot = matched_df[
    ~matched_df["localization"].isin(
        ["mitochondrial", "non-mitochondrial"]
    )
]

In [27]:
# Build a set containing all mitochondrial gene symbols
mito_genes = set(df_mito["Symbol"])

# Add all mitochondrial gene synonyms to the same lookup set
for syns in df_mito["Synonyms"]:

    # Ignore placeholder values
    if syns != "-":

        # Split the synonym list and add each synonym individually
        mito_genes.update(
            s.strip() for s in syns.split("|")
        )

# Select rows that are not already classified as readthrough genes
mask = non_uniprot["localization"] != "readthrough"

# Classify the remaining genes based on whether their gene name
# exists in the mitochondrial gene/synonym set
non_uniprot.loc[mask, "localization"] = (
    non_uniprot.loc[mask, "gene"]
    .apply(
        lambda x:
        "mitochondrial"
        if x in mito_genes
        else "non-mitochondrial"
    )
)

/var/folders/v1/6kddl0v12td99m1lchv02v4m0000gn/T/ipykernel_9928/2058166161.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  non_uniprot.loc[mask, "localization"] = (


In [28]:
# Use gene_id as the index so rows can be aligned during the update
matched_df = matched_df.set_index("gene_id")

# Use gene_id as the index for the updated dataframe as well
non_uniprot = non_uniprot.set_index("gene_id")

# Update matching rows in matched_df with values from non_uniprot
# Only rows with the same gene_id will be modified
matched_df.update(non_uniprot)

# Restore gene_id as a normal dataframe column
matched_df = matched_df.reset_index()

In [29]:
# Remove temporary columns that are no longer needed
matched_df = matched_df.drop(
    columns=[
        "uniprot_id",
        "proteinLength",
        "gene_id"
    ]
)

# Rename the retrieved sequence length column to proteinLength
# so it matches the expected output column name
matched_df = matched_df.rename(
    columns={
        "seq_length": "proteinLength"
    }
)

In [30]:
#reindexing and updating
df = df.set_index("protein_id")
matched_df = matched_df.set_index("protein_id")

df.update(matched_df)

df = df.reset_index()

In [31]:
# Create a subset containing only genes that are still unresolved
# Keep genes classified as either "unclassified" or "length_only"
df_subset = df[
    df["localization"].isin([
        "unclassified",
        "length_only"
    ])
].copy()

In [32]:
# Create a copy of the UniProt dataframe for building the lookup table
tmp = uniprot.copy()


# -------- Expand gene names and synonyms --------

# Combine primary gene names and synonym names into a single string
tmp["match"] = (
    tmp["Gene Names"].fillna("") + " " +
    tmp["Gene Names (synonym)"].fillna("")
)

# Split the combined names into individual identifiers
# and create one row for each identifier
tmp = (
    tmp.assign(match=tmp["match"].str.split())
       .explode("match")
)

# Remove surrounding whitespace from each identifier
tmp["match"] = tmp["match"].str.strip()

# Remove empty identifiers
tmp = tmp[tmp["match"] != ""]


# -------- Merge with unresolved genes --------

# Match unresolved genes against all UniProt gene names and synonyms
m = (
    df_subset.reset_index()
    .merge(
        tmp[
            [
                "match",
                "Localization",
                "Length",
                "Entry",
                "Sequence"
            ]
        ],
        left_on="gene",
        right_on="match",
        how="left",
        suffixes=("", "_uni")
    )
)


# -------- Find the closest protein-length match --------

# Compute the absolute difference between the expected
# protein length and the UniProt protein length
m["len_diff"] = (
    m["Length"] - m["Length_uni"]
).abs()

# For each original gene, keep the UniProt record
# having the smallest length difference
m = (
    m.sort_values("len_diff")
     .drop_duplicates("index")
)


# -------- Filter acceptable matches --------

# Keep the UniProt accession only if the protein lengths
# differ by at most one amino acid
m["Entry"] = m["Entry"].where(
    m["len_diff"] <= 1
)

# Keep the protein sequence only for acceptable matches
m["Sequence"] = m["Sequence"].where(
    m["len_diff"] <= 1
)


# -------- Merge annotations back --------

# Merge the selected UniProt annotation back into the
# unresolved gene dataframe
df_subset = (
    df_subset.reset_index()
    .merge(
        m[
            [
                "index",
                "Localization",
                "Entry",
                "Sequence"
            ]
        ],
        on="index",
        how="left"
    )

    # Remove the temporary index column
    .drop(columns="index")
)

In [33]:
# Select genes that still do not have an assigned protein sequence
missing = df_subset[
    df_subset["Sequence"].isna()
].copy()


# Match these genes against the expanded UniProt gene-name lookup
# to retrieve the corresponding UniProt protein length
missing = (
    missing.merge(
        tmp[
            [
                "match",
                "Length"
            ]
        ],
        left_on="gene",
        right_on="match",
        how="left"
    )

    # Remove the temporary matching column
    .drop(columns="match")
)


# Rename the UniProt protein length column for clarity
missing = missing.rename(
    columns={
        "Length_y": "uniprot_length"
    }
)


# Display the gene name, expected protein length,
# and matched UniProt protein length
missing[
    [
        "gene",
        "Length_x",
        "uniprot_length"
    ]
]

,gene,Length_x,uniprot_length
0,ADAMTS2,253.0,1211.0
1,BCAR1,886.0,870.0
2,DDX39B,328.0,428.0
3,DEAF1,162.0,565.0
4,DLC1,1127.0,89.0
5,DLC1,1127.0,89.0
6,DLC1,1127.0,1528.0
7,DLC1,1127.0,1755.0
8,DLC1,1127.0,1755.0
9,DLGAP2,575.0,1054.0


In [34]:
# Remove existing annotation columns if they are present
# This prevents duplicate columns before renaming
df_subset = df_subset.drop(
    columns=[
        "localization",
        "sequence",
        "UniProt"
    ],
    errors="ignore"
)

# Rename UniProt-derived columns to the desired output names
df_subset = df_subset.rename(
    columns={
        "Localization": "localization",
        "Entry": "UniProt",
        "Sequence": "sequence"
    }
)

# Retain only the final columns required for downstream analysis
df_subset = df_subset[
    [
        "protein_id",      # Protein accession ID
        "gene",            # Gene symbol
        "cds_length",      # Coding sequence length (nt)
        "Length",          # Protein length (aa)
        "localization",    # Mitochondrial/non-mitochondrial annotation
        "proteinLength",   # Protein length from UniProt or matched source
        "UniProt",         # UniProt accession ID
        "sequence"         # Protein amino acid sequence
    ]
]

In [35]:
#reindexing and updating
df = df.set_index("protein_id")

df.update(
    df_subset.set_index("protein_id")
)

df = df.reset_index()

In [36]:
print(df["localization"].value_counts())

non-mitochondrial    14942
mitochondrial         1086
readthrough            141
length_only              1
Name: localization, dtype: int64


In [38]:
unedited = pd.read_csv("all_genes_metadata_without_chx.csv")

In [39]:
df = df.merge(
    unedited[["protein_id", "gene"]]
    .rename(columns={"gene": "gene_old"}),
    on="protein_id",
    how="left"
)

In [46]:
df.to_csv("all_genes_location_without_chx.csv")